In [ ]:
import numpy as np

In [1]:
#single head attention model

import torch
from torch import nn

class model(nn.Module):
  def __init__(
      self,
      img_channels=3,
      base_channels=64,
      time_emb_dim=128,
      time_emb_scale=512,
      num_timesteps=1000,
      num_groups=8
  ):
    super().__init__()

    c1=base_channels
    c2=base_channels*2
    c3=base_channels*4
    c4=base_channels*8

    self.time_embedding=nn.Sequential(
        nn.Embedding(embedding_dim=time_emb_dim, num_embeddings=num_timesteps),
        nn.Linear(in_features=time_emb_dim, out_features=time_emb_scale),
        nn.SiLU(),
        nn.Linear(in_features=time_emb_scale, out_features=time_emb_scale),
    )

    self.conv1=nn.Conv2d(in_channels=img_channels, out_channels=c1, kernel_size=3, stride=1, padding=1)

    #resblock1
    self.res1_1=nn.Sequential(
        nn.GroupNorm(num_groups=num_groups, num_channels=c1),
        nn.SiLU(),
        nn.Conv2d(in_channels=c1, out_channels=c1, kernel_size=3, stride=1, padding=1),
    )

    self.res_time_embed1=nn.Sequential(
        nn.SiLU(),
        nn.Linear(in_features=time_emb_scale, out_features=c1),
    )

    self.res1_2=nn.Sequential(
        nn.GroupNorm(num_groups=num_groups, num_channels=c1),
        nn.SiLU(),
        nn.Conv2d(in_channels=c1, out_channels=c1, kernel_size=3, stride=1, padding=1),
    )

    #down sample 1
    self.down1=nn.Conv2d(in_channels=c1, out_channels=c2, kernel_size=3, stride=2, padding=1)

    #resblock2
    self.res2_1=nn.Sequential(
        nn.GroupNorm(num_groups=num_groups, num_channels=c2),
        nn.SiLU(),
        nn.Conv2d(in_channels=c2, out_channels=c2, kernel_size=3, stride=1, padding=1),
    )

    self.res_time_embed2=nn.Sequential(
        nn.SiLU(),
        nn.Linear(in_features=time_emb_scale, out_features=c2),
    )

    self.res2_2=nn.Sequential(
        nn.GroupNorm(num_groups=num_groups, num_channels=c2),
        nn.SiLU(),
        nn.Conv2d(in_channels=c2, out_channels=c2, kernel_size=3, stride=1, padding=1),
    )

    #down sample 2
    self.down2=nn.Conv2d(in_channels=c2, out_channels=c3, kernel_size=3, stride=2, padding=1)

    #resblock3
    self.res3_1=nn.Sequential(
        nn.GroupNorm(num_groups=num_groups, num_channels=c3),
        nn.SiLU(),
        nn.Conv2d(in_channels=c3, out_channels=c3, kernel_size=3, stride=1, padding=1),
    )

    self.res_time_embed3=nn.Sequential(
        nn.SiLU(),
        nn.Linear(in_features=time_emb_scale, out_features=c3),
    )

    self.res3_2=nn.Sequential(
        nn.GroupNorm(num_groups=num_groups, num_channels=c3),
        nn.SiLU(),
        nn.Conv2d(in_channels=c3, out_channels=c3, kernel_size=3, stride=1, padding=1),
    )

    #down sample 3
    self.down3=nn.Conv2d(in_channels=c3, out_channels=c4, kernel_size=3, stride=2, padding=1)

    #resblock before attention
    self.res_att_1=nn.Sequential(
        nn.GroupNorm(num_groups=num_groups, num_channels=c4),
        nn.SiLU(),
        nn.Conv2d(in_channels=c4, out_channels=c4, kernel_size=3, stride=1, padding=1),
    )

    self.res_time_embed_att=nn.Sequential(
        nn.SiLU(),
        nn.Linear(in_features=time_emb_scale, out_features=c4),
    )

    self.res_att_2=nn.Sequential(
        nn.GroupNorm(num_groups=num_groups, num_channels=c4),
        nn.SiLU(),
        nn.Conv2d(in_channels=c4, out_channels=c4, kernel_size=3, stride=1, padding=1),
    )

    #attention
    self.att_norm=nn.GroupNorm(num_groups, c4)

    self.wq=nn.Linear(in_features=c4, out_features=c4)
    self.wk=nn.Linear(in_features=c4, out_features=c4)
    self.wv=nn.Linear(in_features=c4, out_features=c4)
    self.wo=nn.Linear(in_features=c4, out_features=c4)

    #upconv 1
    self.up1=nn.ConvTranspose2d(in_channels=c4, out_channels=c3, kernel_size=3, stride=2, padding=1, output_padding=1)

    #resblock 1
    self.res4_1=nn.Sequential(
        nn.GroupNorm(num_groups=num_groups, num_channels=c3 * 2),
        nn.SiLU(),
        nn.Conv2d(in_channels=c3 * 2, out_channels=c3, kernel_size=3, stride=1, padding=1),
    )

    self.res_time_embed4=nn.Sequential(
        nn.SiLU(),
        nn.Linear(in_features=time_emb_scale, out_features=c3),
    )

    self.res4_2=nn.Sequential(
        nn.GroupNorm(num_groups=num_groups, num_channels=c3),
        nn.SiLU(),
        nn.Conv2d(in_channels=c3, out_channels=c3, kernel_size=3, stride=1, padding=1),
    )

    self.resx1=nn.Conv2d(in_channels=c3 * 2, out_channels=c3, kernel_size=1)

    #upconv 2
    self.up2=nn.ConvTranspose2d(in_channels=c3, out_channels=c2, kernel_size=3, stride=2, padding=1, output_padding=1)

    #resblock 2
    self.res5_1=nn.Sequential(
        nn.GroupNorm(num_groups=num_groups, num_channels=c2 * 2),
        nn.SiLU(),
        nn.Conv2d(in_channels=c2 * 2, out_channels=c2, kernel_size=3, stride=1, padding=1),
    )

    self.res_time_embed5=nn.Sequential(
        nn.SiLU(),
        nn.Linear(in_features=time_emb_scale, out_features=c2),
    )

    self.res5_2=nn.Sequential(
        nn.GroupNorm(num_groups=num_groups, num_channels=c2),
        nn.SiLU(),
        nn.Conv2d(in_channels=c2, out_channels=c2, kernel_size=3, stride=1, padding=1),
    )

    self.resx2=nn.Conv2d(in_channels=c2 * 2, out_channels=c2, kernel_size=1)

    #upconv 3
    self.up3=nn.ConvTranspose2d(in_channels=c2, out_channels=c1, kernel_size=3, stride=2, padding=1, output_padding=1)

    #resblock 3
    self.res6_1=nn.Sequential(
        nn.GroupNorm(num_groups=num_groups, num_channels=c1 * 2),
        nn.SiLU(),
        nn.Conv2d(in_channels=c1 * 2, out_channels=c1, kernel_size=3, stride=1, padding=1),
    )

    self.res_time_embed6=nn.Sequential(
        nn.SiLU(),
        nn.Linear(in_features=time_emb_scale, out_features=c1),
    )

    self.res6_2=nn.Sequential(
        nn.GroupNorm(num_groups=num_groups, num_channels=c1),
        nn.SiLU(),
        nn.Conv2d(in_channels=c1, out_channels=c1, kernel_size=3, stride=1, padding=1),
    )

    self.resx3=nn.Conv2d(in_channels=c1 * 2, out_channels=c1, kernel_size=1)

    self.conv_f=nn.ConvTranspose2d(in_channels=c1, out_channels=img_channels, kernel_size=3, stride=1, padding=1)

  def forward(self, x, t):
    # Initial Inputs
    # x shape: (B, 3, 32, 32)
    # t shape: (B,)

    t=self.time_embedding(t)
    # t shape: (B, 512)

    xt=self.conv1(x)
    # xt shape: (B, 64, 32, 32)

    # layer 1
    skip1=self.res1_2(self.res1_1(xt)+self.res_time_embed1(t)[:,:,None,None])+xt
    # skip1 shape: (B, 64, 32, 32)
    xt=self.down1(skip1)
    # xt shape: (B, 128, 16, 16)

    # layer 2
    skip2=self.res2_2(self.res2_1(xt)+self.res_time_embed2(t)[:,:,None,None])+xt
    # skip2 shape: (B, 128, 16, 16)
    xt=self.down2(skip2)
    # xt shape: (B, 256, 8, 8)

    # layer 3
    skip3=self.res3_2(self.res3_1(xt)+self.res_time_embed3(t)[:,:,None,None])+xt
    # skip3 shape: (B, 256, 8, 8)
    xt=self.down3(skip3)
    # xt shape: (B, 512, 4, 4)

    # bottleneck - Fixed: Added the missing residual blocks before attention
    xt = self.res_att_2(self.res_att_1(xt) + self.res_time_embed_att(t)[:,:,None,None])+xt
    # xt shape: (B, 512, 4, 4)

    xt=self.att_norm(xt)
    # xt shape: (B, 512, 4, 4)
    x_res=xt
    # x_res shape: (B, 512, 4, 4)

    # Dynamic spatial extraction
    B, C, H_b, W_b = xt.shape
    seq_len = H_b * W_b # 4 * 4 = 16

    xt=xt.flatten(start_dim=2)
    # xt shape: (B, 512, 16)
    xt=xt.permute(0,2,1)
    # xt shape: (B, 16, 512)

    q=self.wq(xt) # q shape: (B, 16, 512)
    k=self.wk(xt) # k shape: (B, 16, 512)
    v=self.wv(xt) # v shape: (B, 16, 512)

    # Single-head attention calculation
    scale_factor = C ** 0.5
    att=torch.einsum('b i d, b j d -> b i j', q, k) / scale_factor
    # att shape: (B, 16, 16)
    att=torch.softmax(att, dim=-1)
    # att shape: (B, 16, 16)

    # Apply attention scores to values
    xt=torch.einsum('b i j, b j d -> b i d', att, v)
    # xt shape: (B, 16, 512)
    
    xt=self.wo(xt)
    # xt shape: (B, 16, 512)
    
    xt=xt.permute(0,2,1)
    # xt shape: (B, 512, 16)
    
    # Restore spatial dimensions dynamically
    xt=xt.unflatten(dim=-1, sizes=(H_b, W_b)) + x_res
    # xt shape: (B, 512, 4, 4)

    # layer 1 (up)
    xt=self.up1(xt)
    # xt shape: (B, 256, 8, 8)
    xt=torch.cat((xt, skip3), dim=1)
    # xt shape: (B, 512, 8, 8)
    xt=self.res4_2(self.res4_1(xt)+self.res_time_embed4(t)[:,:,None,None])+self.resx1(xt)
    # xt shape: (B, 256, 8, 8)

    # layer 2 (up)
    xt=self.up2(xt)
    # xt shape: (B, 128, 16, 16)
    xt=torch.cat((xt, skip2), dim=1)
    # xt shape: (B, 256, 16, 16)
    xt=self.res5_2(self.res5_1(xt)+self.res_time_embed5(t)[:,:,None,None])+self.resx2(xt)
    # xt shape: (B, 128, 16, 16)

    # layer 3 (up)
    xt=self.up3(xt)
    # xt shape: (B, 64, 32, 32)
    xt=torch.cat((xt, skip1), dim=1)
    # xt shape: (B, 128, 32, 32)
    xt=self.res6_2(self.res6_1(xt)+self.res_time_embed6(t)[:,:,None,None])+self.resx3(xt)
    # xt shape: (B, 64, 32, 32)

    # output
    xt=self.conv_f(xt)
    # xt shape: (B, 3, 32, 32)

    return xt

In [ ]:
import torch
from torch import nn

class model2(nn.Module):
  def __init__(
      self, 
      img_channels=1,
      base_channels=64,
      time_emb_dim=128,
      time_emb_scale=512,
      num_timesteps=1000,
      num_groups=8,
      num_heads=8
  ):
    super().__init__()

    self.num_heads=num_heads
    
    c1=base_channels
    c2=base_channels*2
    c3=base_channels*4
    c4=base_channels*8
    
    self.time_embedding=nn.Sequential(
        nn.Embedding(embedding_dim=time_emb_dim, num_embeddings=num_timesteps),
        nn.Linear(in_features=time_emb_dim, out_features=time_emb_scale),
        nn.SiLU(),
        nn.Linear(in_features=time_emb_scale, out_features=time_emb_scale),
    )

    self.conv1=nn.Conv2d(in_channels=img_channels, out_channels=c1, kernel_size=3, stride=1, padding=1)

    #resblock1
    self.res1_1=nn.Sequential(
        nn.GroupNorm(num_groups=num_groups, num_channels=c1),
        nn.SiLU(),
        nn.Conv2d(in_channels=c1, out_channels=c1, kernel_size=3, stride=1, padding=1),
    )

    self.res_time_embed1=nn.Sequential(
        nn.SiLU(),
        nn.Linear(in_features=time_emb_scale, out_features=c1),
    )

    self.res1_2=nn.Sequential(
        nn.GroupNorm(num_groups=num_groups, num_channels=c1),
        nn.SiLU(),
        nn.Conv2d(in_channels=c1, out_channels=c1, kernel_size=3, stride=1, padding=1),
    )

    #down sample 1
    self.down1=nn.Conv2d(in_channels=c1, out_channels=c2, kernel_size=3, stride=2, padding=1)

    #resblock2
    self.res2_1=nn.Sequential(
        nn.GroupNorm(num_groups=num_groups, num_channels=c2),
        nn.SiLU(),
        nn.Conv2d(in_channels=c2, out_channels=c2, kernel_size=3, stride=1, padding=1),
    )

    self.res_time_embed2=nn.Sequential(
        nn.SiLU(),
        nn.Linear(in_features=time_emb_scale, out_features=c2),
    )

    self.res2_2=nn.Sequential(
        nn.GroupNorm(num_groups=num_groups, num_channels=c2),
        nn.SiLU(),
        nn.Conv2d(in_channels=c2, out_channels=c2, kernel_size=3, stride=1, padding=1),
    )

    #down sample 2
    self.down2=nn.Conv2d(in_channels=c2, out_channels=c3, kernel_size=3, stride=2, padding=1)

    #resblock3
    self.res3_1=nn.Sequential(
        nn.GroupNorm(num_groups=num_groups, num_channels=c3),
        nn.SiLU(),
        nn.Conv2d(in_channels=c3, out_channels=c3, kernel_size=3, stride=1, padding=1),
    )

    self.res_time_embed3=nn.Sequential(
        nn.SiLU(),
        nn.Linear(in_features=time_emb_scale, out_features=c3),
    )

    self.res3_2=nn.Sequential(
        nn.GroupNorm(num_groups=num_groups, num_channels=c3),
        nn.SiLU(),
        nn.Conv2d(in_channels=c3, out_channels=c3, kernel_size=3, stride=1, padding=1),
    )

    #down sample 3
    self.down3=nn.Conv2d(in_channels=c3, out_channels=c4, kernel_size=3, stride=2, padding=1)

    #resblock before attention
    self.res_att_1=nn.Sequential(
        nn.GroupNorm(num_groups=num_groups, num_channels=c4),
        nn.SiLU(),
        nn.Conv2d(in_channels=c4, out_channels=c4, kernel_size=3, stride=1, padding=1),
    )

    self.res_time_embed_att=nn.Sequential(
        nn.SiLU(),
        nn.Linear(in_features=time_emb_scale, out_features=c4),
    )

    self.res_att_2=nn.Sequential(
        nn.GroupNorm(num_groups=num_groups, num_channels=c4),
        nn.SiLU(),
        nn.Conv2d(in_channels=c4, out_channels=c4, kernel_size=3, stride=1, padding=1),
    )

    #attention
    self.att_norm=nn.GroupNorm(num_groups, c4)

    self.wq=nn.Linear(in_features=c4, out_features=c4)
    self.wk=nn.Linear(in_features=c4, out_features=c4)
    self.wv=nn.Linear(in_features=c4, out_features=c4)
    self.wo=nn.Linear(in_features=c4, out_features=c4)

    #upconv 1
    self.up1=nn.ConvTranspose2d(in_channels=c4, out_channels=c3, kernel_size=3, stride=2, padding=1, output_padding=1)

    #resblock 1 (in_channels doubled due to skip connection concat)
    self.res4_1=nn.Sequential(
        nn.GroupNorm(num_groups=num_groups, num_channels=c3 * 2),
        nn.SiLU(),
        nn.Conv2d(in_channels=c3 * 2, out_channels=c3, kernel_size=3, stride=1, padding=1),
    )

    self.res_time_embed4=nn.Sequential(
        nn.SiLU(),
        nn.Linear(in_features=time_emb_scale, out_features=c3),
    )

    self.res4_2=nn.Sequential(
        nn.GroupNorm(num_groups=num_groups, num_channels=c3),
        nn.SiLU(),
        nn.Conv2d(in_channels=c3, out_channels=c3, kernel_size=3, stride=1, padding=1),
    )

    self.resx1=nn.Conv2d(in_channels=c3 * 2, out_channels=c3, kernel_size=1)

    #upconv 2
    self.up2=nn.ConvTranspose2d(in_channels=c3, out_channels=c2, kernel_size=3, stride=2, padding=1, output_padding=1)

    #resblock 2
    self.res5_1=nn.Sequential(
        nn.GroupNorm(num_groups=num_groups, num_channels=c2 * 2),
        nn.SiLU(),
        nn.Conv2d(in_channels=c2 * 2, out_channels=c2, kernel_size=3, stride=1, padding=1),
    )

    self.res_time_embed5=nn.Sequential(
        nn.SiLU(),
        nn.Linear(in_features=time_emb_scale, out_features=c2),
    )

    self.res5_2=nn.Sequential(
        nn.GroupNorm(num_groups=num_groups, num_channels=c2),
        nn.SiLU(),
        nn.Conv2d(in_channels=c2, out_channels=c2, kernel_size=3, stride=1, padding=1),
    )

    self.resx2=nn.Conv2d(in_channels=c2 * 2, out_channels=c2, kernel_size=1)

    #upconv 3
    self.up3=nn.ConvTranspose2d(in_channels=c2, out_channels=c1, kernel_size=3, stride=2, padding=1, output_padding=1)

    #resblock 3
    self.res6_1=nn.Sequential(
        nn.GroupNorm(num_groups=num_groups, num_channels=c1 * 2),
        nn.SiLU(),
        nn.Conv2d(in_channels=c1 * 2, out_channels=c1, kernel_size=3, stride=1, padding=1),
    )

    self.res_time_embed6=nn.Sequential(
        nn.SiLU(),
        nn.Linear(in_features=time_emb_scale, out_features=c1),
    )

    self.res6_2=nn.Sequential(
        nn.GroupNorm(num_groups=num_groups, num_channels=c1),
        nn.SiLU(),
        nn.Conv2d(in_channels=c1, out_channels=c1, kernel_size=3, stride=1, padding=1),
    )

    self.resx3=nn.Conv2d(in_channels=c1 * 2, out_channels=c1, kernel_size=1)

    self.conv_f=nn.ConvTranspose2d(in_channels=c1, out_channels=img_channels, kernel_size=3, stride=1, padding=1)

  def forward(self, x, t):
    # Initial Inputs
    # x shape: (B, 1, 32, 32)
    # t shape: (B,)
    
    t=self.time_embedding(t) 
    # t shape: (B, 512)

    xt=self.conv1(x) 
    # xt shape: (B, 64, 32, 32)

    # layer 1
    skip1=self.res1_2(self.res1_1(xt)+self.res_time_embed1(t)[:,:,None,None])+xt 
    # skip1 shape: (B, 64, 32, 32)
    xt=self.down1(skip1) 
    # xt shape: (B, 128, 16, 16)

    # layer 2
    skip2=self.res2_2(self.res2_1(xt)+self.res_time_embed2(t)[:,:,None,None])+xt 
    # skip2 shape: (B, 128, 16, 16)
    xt=self.down2(skip2) 
    # xt shape: (B, 256, 8, 8)

    # layer 3
    skip3=self.res3_2(self.res3_1(xt)+self.res_time_embed3(t)[:,:,None,None])+xt 
    # skip3 shape: (B, 256, 8, 8)
    xt=self.down3(skip3) 
    # xt shape: (B, 512, 4, 4)

    # bottleneck pre-attention
    xt = self.res_att_2(self.res_att_1(xt) + self.res_time_embed_att(t)[:,:,None,None])+xt
    # xt shape: (B, 512, 4, 4)

    # attention prep
    xt=self.att_norm(xt) 
    # xt shape: (B, 512, 4, 4)
    x_res=xt 
    # x_res shape: (B, 512, 4, 4)
    
    B, C, H_b, W_b=xt.shape
    seq_len=H_b*W_b # 4 * 4 = 16
    head_dim=C//self.num_heads # 512 // 8 = 64

    xt=xt.flatten(start_dim=2) 
    # xt shape: (B, 512, 16)
    xt=xt.permute(0,2,1) 
    # xt shape: (B, 16, 512)

    q=self.wq(xt) # q shape: (B, 16, 512)
    k=self.wk(xt) # k shape: (B, 16, 512)
    v=self.wv(xt) # v shape: (B, 16, 512)

    # reshape for multi-head setup
    q=q.reshape(B, seq_len, self.num_heads, head_dim) # q shape: (B, 16, 8, 64)
    k=k.reshape(B, seq_len, self.num_heads, head_dim) # k shape: (B, 16, 8, 64)
    v=v.reshape(B, seq_len, self.num_heads, head_dim) # v shape: (B, 16, 8, 64)

    # permute for scaled dot-product attention mapping
    q=q.permute(0, 2, 1, 3) # q shape: (B, 8, 16, 64)
    k=k.permute(0, 2, 1, 3) # k shape: (B, 8, 16, 64)
    v=v.permute(0, 2, 1, 3) # v shape: (B, 8, 16, 64)

    # compute attention matrix
    scale_factor=head_dim**0.5 
    att=torch.einsum('b h q d, b h k d -> b h q k', q, k) / scale_factor 
    # att shape: (B, 8, 16, 16)
    att=torch.softmax(att, dim=-1) 
    # att shape: (B, 8, 16, 16)

    # apply attention to values
    xt=torch.einsum('b h q k, b h k d -> b h q d', att, v) 
    # xt shape: (B, 8, 16, 64)
    
    # collapse heads back down
    xt=xt.permute(0, 2, 1, 3) 
    # xt shape: (B, 16, 8, 64)
    xt=xt.reshape(B, seq_len, C) 
    # xt shape: (B, 16, 512)
    
    xt=self.wo(xt) 
    # xt shape: (B, 16, 512)
    
    # restore spatial dimensions
    xt=xt.permute(0,2,1) 
    # xt shape: (B, 512, 16)
    xt=xt.unflatten(dim=-1, sizes=(H_b, W_b)) + x_res 
    # xt shape: (B, 512, 4, 4)

    # layer 1 (up)
    xt=self.up1(xt) 
    # xt shape: (B, 256, 8, 8)
    xt=torch.cat((xt, skip3), dim=1) 
    # xt shape: (B, 512, 8, 8) -- (256 from up1 + 256 from skip3)
    xt=self.res4_2(self.res4_1(xt)+self.res_time_embed4(t)[:,:,None,None])+self.resx1(xt) 
    # xt shape: (B, 256, 8, 8)

    # layer 2 (up)
    xt=self.up2(xt) 
    # xt shape: (B, 128, 16, 16)
    xt=torch.cat((xt, skip2), dim=1) 
    # xt shape: (B, 256, 16, 16) -- (128 from up2 + 128 from skip2)
    xt=self.res5_2(self.res5_1(xt)+self.res_time_embed5(t)[:,:,None,None])+self.resx2(xt) 
    # xt shape: (B, 128, 16, 16)

    # layer 3 (up)
    xt=self.up3(xt) 
    # xt shape: (B, 64, 32, 32)
    xt=torch.cat((xt, skip1), dim=1) 
    # xt shape: (B, 128, 32, 32) -- (64 from up3 + 64 from skip1)
    xt=self.res6_2(self.res6_1(xt)+self.res_time_embed6(t)[:,:,None,None])+self.resx3(xt) 
    # xt shape: (B, 64, 32, 32)

    # output
    out=self.conv_f(xt) 
    # out shape: (B, 1, 32, 32)

    return out

In [2]:
# device="cuda" if torch.cuda.is_available() else "cpu"

In [3]:
import torchvision
import torchvision.transforms as transforms

#originally this code was written for cifar10 which are 32x32 but then i switched to mnist which are 28x28 that is why padding is need to make it 32x32
transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        [0.5, 0.5, 0.5],#mean
        [0.5, 0.5, 0.5]#std
    )
])

trainset=torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

trainloader=torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True)

100%|██████████| 170M/170M [00:20<00:00, 8.17MB/s] 


In [4]:
import copy

#exponential moving average: as the weight change might be abrupt (may move away from the optimal weights).this function ensures smooth transition for weights

@torch.no_grad()
def update_ema(ema_model, model, decay=0.995):
  ema_params=dict(ema_model.named_parameters())
  model_params=dict(model.named_parameters())

  for k in ema_params.keys():
    ema_params[k].data.mul_(decay).add_(model_params[k].data, alpha=1-decay)

In [5]:
from accelerate import Accelerator

accelerator=Accelerator(mixed_precision="fp16")

In [6]:
import random

def train(
    ddpm,
    trainloader,
    alpha_bar,
    ddpm_model_name,
    ema_model_name=False,
    ema_model=False
):
    
    if ema_model is not False:
        ema_model.to(accelerator.device)
        ema_model.eval()
        
    optimizer=torch.optim.AdamW(params=ddpm.parameters(), lr=2e-4)
    loss_fn=nn.MSELoss()
    
    losses=[]
    
    
    epochs=500

    ddpm, optimizer, trainloader=accelerator.prepare(ddpm, optimizer, trainloader)
    alpha_bar=alpha_bar.to(accelerator.device)
    
    ddpm.train()
    for epoch in range(epochs):
      total_loss=0
    
      for X, y in trainloader:
    
        t=torch.randint(0, 1000, (X.shape[0],), device=accelerator.device)
    
        e=torch.randn_like(X)
        xt=torch.sqrt(alpha_bar[t])[:,None,None,None]*X+torch.sqrt(1-alpha_bar[t])[:,None,None,None]*e
        e_pred=ddpm(xt, t)
    
        optimizer.zero_grad()

        loss=loss_fn(e, e_pred)
    
        accelerator.backward(loss)
        optimizer.step()

        if ema_model is not False:
            update_ema(ema_model, ddpm)
        total_loss+=loss.item()
    
      avg_loss=total_loss/len(trainloader)
        
      losses.append(avg_loss)

      torch.save(ddpm.state_dict(), ddpm_model_name)  
      if ema_model is not False:  
          torch.save(ema_model.state_dict(), ema_model_name)
          
      print(f"epoch={epoch}; avg_loss={avg_loss}")

    return losses

In [19]:
import matplotlib.pyplot as plt
def sample_model(ddpm, alpha, alpha_bar, posterior_variance, ema_model=False):
    if ema_model is False:
        ema_model=ddpm

    ddpm.to(accelerator.device)
    
    ema_model.eval()
    
    alpha=alpha.to(accelerator.device)
    alpha_bar=alpha_bar.to(accelerator.device)
    posterior_variance=posterior_variance.to(accelerator.device)
    
    t=999
    
    xt=torch.randn((1,3,32,32), device=accelerator.device)
    
    samples=[]
    
    with torch.no_grad():
      while t>=0:
        t_tensor=torch.tensor([t], device=accelerator.device)
        e_pred=ema_model(xt, t_tensor)
    
        if t>0:
          z=torch.randn_like(xt)
        else:
          z=torch.zeros_like(xt)
    
        #here we use posterior variance instead of beta as this is reverse process
        xt=(1/torch.sqrt(alpha[t]))*(xt-((1-alpha[t])/torch.sqrt(1-alpha_bar[t]))*e_pred)+torch.sqrt(posterior_variance[t])*z
    
        #print(f"t={t}; mean={xt.mean().item():.4f}; std={xt.std().item():.4f}")
    
        if t%100==0:
          samples.append(xt[0].detach().cpu())
    
        t=t-1
    
    #denoising trajectory visualization
    fig, axes=plt.subplots(1, len(samples), figsize=(20,3))
    
    for i, sample in enumerate(samples):
      img=sample.permute(1,2,0)
      img=(img+1)/2
      img=img.clamp(0,1)
      axes[i].imshow(img.numpy())
      axes[i].axis("off")
      axes[i].set_title(f"step {i}")
    
    plt.show()


In [8]:
#linear beta scheduling

timesteps = 1000

linear_beta=torch.linspace(
    1e-4,
    0.02,
    timesteps
)

linear_alpha=1.0-linear_beta

linear_alpha_bar=torch.cumprod(linear_alpha, dim=0)

#how uncertain are we about xt-1 given xt
linear_posterior_variance=linear_beta.clone()

linear_posterior_variance[1:]=(linear_beta[1:]*(1-linear_alpha_bar[:-1])/(1-linear_alpha_bar[1:]))

In [ ]:
#cosine beta scheduling

import math

timesteps=1000

s=0.008

steps=timesteps+1

x=torch.linspace(
    0,
    timesteps,
    steps,
)

cosine_alpha_bar=torch.cos(((x/timesteps+s)/(1+s))*math.pi*0.5)**2

cosine_alpha_bar=cosine_alpha_bar/cosine_alpha_bar[0]

cosine_beta=1-(cosine_alpha_bar[1:]/cosine_alpha_bar[:-1])

cosine_beta=torch.clamp(cosine_beta, 1e-4, 0.999)

cosine_alpha=1.0-cosine_beta

cosine_alpha_bar=torch.cumprod(cosine_alpha, dim=0)

#how uncertain are we about xt-1 given xt
cosine_posterior_variance=cosine_beta.clone()

cosine_posterior_variance[1:]=(cosine_beta[1:]*(1-cosine_alpha_bar[:-1])/(1-cosine_alpha_bar[1:]))


In [ ]:
def save(name, arr):
    np.save(name, np.array(arr))

Baseline(model1):

linear beta

no ema

single head attention

In [16]:
model1=model()

In [11]:
losses1=train(model1, trainloader, linear_alpha_bar, 'model1')
save('losses1.npy', losses1)

model2:

cosine beta

no ema

single head

In [ ]:
model2=model()

In [ ]:
losses2=train(model2, trainloader, cosine_alpha_bar, 'model2')
save('losses2.npy', losses2)

model3:

cosine beta

ema

single head

In [ ]:
model3=model()

In [ ]:
ema_model3=copy.deepcopy(model3)
losses3=train(model3, trainloader, cosine_alpha_bar, 'model3', 'ema_model3', ema_model3)
save('losses3.npy', losses3)

model4:

cosine beta

ema

multihead

In [ ]:
model4=model2()

In [ ]:
ema_model4=copy.deepcopy(model4)
losses4=train(model4, trainloader, cosine_alpha_bar, 'model4', 'ema_model4', ema_model4)
save('losses4.npy', losses4)

In [ ]:
# They will ask you exactly how PyTorch handles gradients across two GPUs over the PCIe bus.

# They will ask you what "underflow" is and how the GradScaler mathematically fixes it.

# They will ask you to explain memory pointers and memory leaks in your C++ code.